In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

MU = 1.0
X0 = 1.0
V0 = 0.0

LAYERS = (1, 32, 32, 32, 32, 1)
WINDOW_SIZE = 2.0

N_COLLOCATION_PER_WINDOW = 2000
ADAM_EPOCHS = 3000
ADAM_LR = 1e-3
LBFGS_MAX_ITER = 2000
PRINT_EVERY = 500

PERIOD_MARGIN = 1.15
ROUGH_PERIOD_FACTOR = 1.614
N_PERIODS_SETTLE = 4

In [ ]:
def van_der_pol_rhs(t, y, mu=MU):
    x, v = y
    return [v, mu * (1 - x**2) * v - x]

def estimate_vdp_period(mu, x0=X0, v0=V0, rough_factor=ROUGH_PERIOD_FACTOR, n_settle=N_PERIODS_SETTLE):

    rough_T = rough_factor * mu
    t_span = n_settle * max(rough_T, 5.0)

    def event_peak(t, y):
        return y[1]
    event_peak.direction = -1

    sol = solve_ivp(
        lambda t, y: van_der_pol_rhs(t, y, mu=mu),
        [0.0, t_span], [x0, v0], events=event_peak,
        rtol=1e-10, atol=1e-12, max_step=0.05
    )
    peak_times = sol.t_events[0]
    if len(peak_times) < 3:
        print(f"[period detection] mu={mu}: not enough peaks found in the settling "
              f"window, falling back to rough estimate {rough_T:.3f}")
        return rough_T
    return peak_times[-1] - peak_times[-2]

vdp_period = estimate_vdp_period(MU)
T_FINAL = float(PERIOD_MARGIN * vdp_period)
print(f"Detected VdP period at mu={MU}: {vdp_period:.4f} -> T_FINAL = {T_FINAL:.4f}")

In [ ]:
class BasePINN(nn.Module):
    def __init__(self):
        super().__init__()

    def encode_input(self, t):
        raise NotImplementedError("Subclasses must implement encode_input().")

    def trunk_forward(self, encoded):
        raise NotImplementedError("Subclasses must implement trunk_forward().")

    def forward(self, t):
        encoded = self.encode_input(t)
        return self.trunk_forward(encoded)


class SirenPINN(BasePINN):
    # SIREN PINN variant
    def __init__(self, t_start, window_size=WINDOW_SIZE, layers=LAYERS, omega_0=30.0):
        super().__init__()
        self.t_start = t_start
        self.window_size = window_size
        self.omega_0 = omega_0

        self.trunk = nn.ModuleList()
        for i in range(len(layers) - 1):
            linear = nn.Linear(layers[i], layers[i + 1])
            with torch.no_grad():
                if i == 0:
                    linear.weight.uniform_(-1 / layers[i], 1 / layers[i])
                else:
                    c = np.sqrt(6 / layers[i]) / self.omega_0
                    linear.weight.uniform_(-c, c)
                linear.bias.zero_()
            self.trunk.append(linear)

    def encode_input(self, t):
        tau = t - self.t_start
        return (2.0 * tau / self.window_size) - 1.0

    def trunk_forward(self, encoded):
        out = encoded
        for layer in self.trunk[:-1]:
            out = torch.sin(self.omega_0 * layer(out))
        return self.trunk[-1](out)


class WindowPINN(nn.Module):

    def __init__(self, base_model, t_start, window_size, x0_local, v0_local):
        super().__init__()
        self.base_model = base_model
        self.t_start = t_start
        self.window_size = window_size
        self.x0_local = x0_local
        self.v0_local = v0_local

    def forward(self, t):
        tau = t - self.t_start
        n_raw = self.base_model(t)
        return self.x0_local + self.v0_local * tau + (tau ** 2) * n_raw


In [ ]:
def physics_residual(model, t, mu=MU):
    """Calculates the differential equation residual using autograd."""
    x = model(t)
    x_t = torch.autograd.grad(x, t, grad_outputs=torch.ones_like(x), create_graph=True, retain_graph=True)[0]
    x_tt = torch.autograd.grad(x_t, t, grad_outputs=torch.ones_like(x_t), create_graph=True, retain_graph=True)[0]
    return x_tt - mu * (1.0 - x ** 2) * x_t + x

def compute_loss(model, t_f):
    # IC is hard-enforced by the WindowPINN wrapper
    f_pred = physics_residual(model, t_f)
    return torch.mean(f_pred ** 2)

def sample_collocation_points(t_start, t_end, n):
    t_f = np.random.uniform(low=t_start, high=t_end, size=(n, 1))
    t_f = torch.tensor(t_f, dtype=torch.get_default_dtype(), device=device)
    t_f.requires_grad_(True)
    return t_f

def evaluate_state(model, t_value):
    t_tensor = torch.tensor([[t_value]], dtype=torch.get_default_dtype(), device=device, requires_grad=True)
    x = model(t_tensor)
    x_t = torch.autograd.grad(x, t_tensor, grad_outputs=torch.ones_like(x))[0]
    return x.item(), x_t.item()


In [ ]:
OMEGA_0 = 10.0

In [ ]:
n_windows = int(np.ceil(T_FINAL / WINDOW_SIZE))
print(f"Total time {T_FINAL:.2f} / window_size {WINDOW_SIZE} -> {n_windows} windows")

def train_window(model, t_f, adam_epochs=ADAM_EPOCHS, adam_lr=ADAM_LR,
                  lbfgs_max_iter=LBFGS_MAX_ITER, print_every=PRINT_EVERY, window_idx=0):
    history = []
    optimizer_adam = torch.optim.Adam(model.parameters(), lr=adam_lr)
    for epoch in range(adam_epochs):
        optimizer_adam.zero_grad()
        loss = compute_loss(model, t_f)
        loss.backward()
        optimizer_adam.step()
        history.append(loss.item())
        if epoch % print_every == 0:
            print(f"[window {window_idx}] [Adam]   epoch {epoch:5d} | loss {loss.item():.3e}")

    optimizer_lbfgs = torch.optim.LBFGS(
        model.parameters(), max_iter=lbfgs_max_iter,
        tolerance_grad=1e-9, tolerance_change=1e-12,
        history_size=50, line_search_fn="strong_wolfe",
    )
    counter = {"n": 0}

    def closure():
        optimizer_lbfgs.zero_grad()
        loss = compute_loss(model, t_f)
        loss.backward()
        history.append(loss.item())
        counter["n"] += 1
        if counter["n"] % print_every == 0:
            print(f"[window {window_idx}] [L-BFGS] iter  {counter['n']:5d} | loss {loss.item():.3e}")
        return loss

    optimizer_lbfgs.step(closure)
    return history

In [ ]:
start_time = time.time()
models = []
full_history = []
x0_cur, v0_cur = X0, V0

for w in range(n_windows):
    t_start = w * WINDOW_SIZE
    t_end = min((w + 1) * WINDOW_SIZE, T_FINAL)
    current_window_size = t_end - t_start

    print(f"\n--- Training Window {w+1}/{n_windows} [t={t_start:.2f} to {t_end:.2f}] ---")

    base = SirenPINN(t_start, window_size=current_window_size, omega_0=OMEGA_0).to(device)
    model_w = WindowPINN(base, t_start, current_window_size, x0_cur, v0_cur).to(device)

    if w > 0:
        model_w.base_model.load_state_dict(models[-1].base_model.state_dict())

    t_f = sample_collocation_points(t_start, t_end, N_COLLOCATION_PER_WINDOW)
    hist = train_window(model_w, t_f, window_idx=w)
    full_history.extend(hist)

    x0_cur, v0_cur = evaluate_state(model_w, t_end)
    models.append(model_w)

training_time = time.time() - start_time
print(f"\nTime-Marching Training Complete! ({training_time:.1f}s)")

In [ ]:
t_eval = np.linspace(0, T_FINAL, 2000)
sol = solve_ivp(van_der_pol_rhs, [0, T_FINAL], [X0, V0], t_eval=t_eval, method="RK45", rtol=1e-10, atol=1e-10)
x_reference = sol.y[0]
v_reference = sol.y[1]

x_pred = np.zeros_like(t_eval)
for w, model_w in enumerate(models):
    t_start = w * WINDOW_SIZE
    t_end = min((w + 1) * WINDOW_SIZE, T_FINAL)
    mask = (t_eval >= t_start) & (t_eval <= t_end + 1e-9)
    if not np.any(mask):
        continue
    t_local = torch.tensor(t_eval[mask].reshape(-1, 1), dtype=torch.get_default_dtype(), device=device)
    with torch.no_grad():
        x_pred[mask] = model_w(t_local).cpu().numpy().flatten()

mse = np.mean((x_pred - x_reference) ** 2)
relative_l2_error = np.linalg.norm(x_pred - x_reference) / np.linalg.norm(x_reference)
final_training_loss = full_history[-1]
n_params = sum(p.numel() for p in models[0].parameters())

# Metrics parity with the adaptive-window script
mae = np.mean(np.abs(x_pred - x_reference))
max_error = np.max(np.abs(x_pred - x_reference))
rmse = np.sqrt(mse)

t_res_test = torch.tensor(np.random.uniform(0.0, T_FINAL, size=(2000, 1)), dtype=torch.get_default_dtype(), device=device)
residual_vals = []
for w, model_w in enumerate(models):
    t_start = w * WINDOW_SIZE
    t_end = min((w + 1) * WINDOW_SIZE, T_FINAL)
    mask_np = (t_res_test.detach().cpu().numpy().flatten() >= t_start) & (t_res_test.detach().cpu().numpy().flatten() <= t_end)
    if not np.any(mask_np):
        continue
    t_sub = t_res_test[mask_np].clone().detach().requires_grad_(True)
    f_sub = physics_residual(model_w, t_sub)
    residual_vals.append(f_sub.detach().cpu().numpy())
mean_physics_residual = np.mean(np.concatenate(residual_vals) ** 2) if residual_vals else float("nan")

print(f"MU: {MU}")
print(f"T_FINAL (auto, {PERIOD_MARGIN}x detected period {vdp_period:.4f}): {T_FINAL:.4f}")
print(f"MSE: {mse:.3e}")
print(f"Relative L2 error: {relative_l2_error:.3e}")
print(f"Final training loss: {final_training_loss:.3e}")
print(f"Mean physics residual (test points): {mean_physics_residual:.3e}")
print(f"Training time: {training_time:.1f}s")
print(f"Trainable parameters (per window model): {n_params}")
print(f"Windows: {n_windows}")
print(f"MAE: {mae:.3e}")
print(f"Max Error (L_inf): {max_error:.3e}")
print(f"RMSE: {rmse:.3e}")
print(f"omega_0 used: {OMEGA_0}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].plot(t_eval, x_reference, "b-", linewidth=2, label="Reference (RK45)")
axes[0, 0].plot(t_eval, x_pred, "r--", linewidth=2, label="SIREN PINN (time-marched)")
for w in range(1, n_windows):
    axes[0, 0].axvline(w * WINDOW_SIZE, color="gray", linestyle=":", linewidth=0.7)
axes[0, 0].set_xlabel("t")
axes[0, 0].set_ylabel("x(t)")
axes[0, 0].set_title("Van der Pol oscillator: SIREN PINN vs. reference")
axes[0, 0].legend()

axes[0, 1].semilogy(full_history, label="physics residual loss")
axes[0, 1].set_xlabel("iteration (concatenated across windows)")
axes[0, 1].set_ylabel("loss (log scale)")
axes[0, 1].set_title("Training loss history")
axes[0, 1].legend()

axes[1, 0].plot(t_eval, np.abs(x_pred - x_reference), "k-")
for w in range(1, n_windows):
    axes[1, 0].axvline(w * WINDOW_SIZE, color="gray", linestyle=":", linewidth=0.7)
axes[1, 0].set_xlabel("t")
axes[1, 0].set_ylabel("|x_pred - x_reference|")
axes[1, 0].set_title("Absolute error")

x_pred_dot = np.gradient(x_pred, t_eval)
axes[1, 1].plot(x_reference, v_reference, "b-", linewidth=2, label="Reference")
axes[1, 1].plot(x_pred, x_pred_dot, "r--", linewidth=2, label="SIREN PINN")
axes[1, 1].set_xlabel("x")
axes[1, 1].set_ylabel("x'")
axes[1, 1].set_title("Phase portrait")
axes[1, 1].legend()

plt.tight_layout()
plt.show()